<a href="https://colab.research.google.com/github/asad-raza-929/Flyrank_Internship_ml/blob/main/Copy_of_w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/asad-raza-929/Flyrank_Internship_ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

My rule is designed to prioritize customers for an 'action' (e.g., a special offer) based on two primary factors: how recently they last engaged and their average purchase value. The assumption is that customers who have engaged more recently and spend more on average are more likely to respond positively to an intervention.

**Rule in plain words:** Customers are scored based on a combination of their recency of activity and their average purchase value. Higher scores are given to customers with more recent activity and higher average purchase values.

**Reason Codes:**
- `RECENT_HIGH_VALUE`: Customer shows recent activity (e.g., within the last 30 days) and has an average purchase value above a certain threshold.
- `RECENT_LOW_VALUE`: Customer shows recent activity but their average purchase value is below the threshold.
- `OLD_HIGH_VALUE`: Customer's last activity was not recent (e.g., more than 30 days ago) but they have an average purchase value above the threshold.
- `OLD_LOW_VALUE`: Customer's last activity was not recent and their average purchase value is below the threshold.
- `NEW_CUSTOMER`: Customer is new and doesn't have enough history for the other classifications yet. (This will be assumed for demonstration purposes only if a customer has no activity at all.)

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [ ]:
import pandas as pd
import numpy as np
import os

# Define parameters for the baseline rule
RECENT_ACTIVITY_THRESHOLD_DAYS = 30
AVERAGE_PURCHASE_VALUE_THRESHOLD = 50.0

print(f"Recency threshold: {RECENT_ACTIVITY_THRESHOLD_DAYS} days")
print(f"Average purchase value threshold: ${AVERAGE_PURCHASE_VALUE_THRESHOLD}")

Recency threshold: 30 days
Average purchase value threshold: $50.0


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [ ]:
# Load the ranked queue from the CSV file
output_csv_path = 'work/outputs/baseline_action_score.csv'
df_ranked = pd.read_csv(output_csv_path)

# Display the top customers (e.g., top 5, or all if less than 5)
print("Top ranked customers from baseline_action_score.csv:")
display(df_ranked.head(min(len(df_ranked), 5)))

Top ranked customers from baseline_action_score.csv:


,customer_id,last_activity_date,average_purchase_value,recency_days,action_score,reason_code,rank
0,19,2023-10-08,100.0,18,100,RECENT_HIGH_VALUE,1
1,9,2023-10-18,90.0,8,100,RECENT_HIGH_VALUE,1
2,6,2023-10-10,80.0,16,100,RECENT_HIGH_VALUE,1
3,14,2023-10-05,70.0,21,100,RECENT_HIGH_VALUE,1
4,1,2023-10-20,60.0,6,100,RECENT_HIGH_VALUE,1


### Top Picks Analysis:

Based on the ranked queue, here's an analysis of the top picks:

**Action:** For customers with high action scores (e.g., `RECENT_HIGH_VALUE` and `RECENT_LOW_VALUE`), the recommended action would be a targeted engagement campaign, such as a special offer or personalized communication.

**Reason Codes and Confidence Notes:**

-   **Customer ID 1 (RECENT_HIGH_VALUE):**
    -   **Action:** High-priority personalized offer.
    -   **Reason Code:** `RECENT_HIGH_VALUE` (Recency: 6 days, Avg Purchase: $60.0).
    -   **Confidence Note:** High confidence. This customer has very recent activity and a strong average purchase value, indicating high engagement and potential for positive response.
    -   **What would make it wrong:** If their last purchase was a one-off anomaly, or if their recent activity was driven by a non-purchase interaction (e.g., a customer service query).

-   **Customer ID 14 (RECENT_HIGH_VALUE):**
    -   **Action:** High-priority personalized offer.
    -   **Reason Code:** `RECENT_HIGH_VALUE` (Recency: 21 days, Avg Purchase: $70.0).
    -   **Confidence Note:** High confidence. Similar to Customer 1, recent activity combined with a good average purchase value makes this a strong candidate.
    -   **What would make it wrong:** If their recent activity was due to a return or complaint, or if the product they typically purchase is out of stock.

-   **Customer ID 6 (RECENT_HIGH_VALUE):**
    -   **Action:** High-priority personalized offer.
    -   **Reason Code:** `RECENT_HIGH_VALUE` (Recency: 16 days, Avg Purchase: $80.0).
    -   **Confidence Note:** High confidence. Strong recent activity and a high average purchase value.
    -   **What would make it wrong:** If the 'average purchase value' is inflated by a single large purchase that is not representative of their usual spending habits.

-   **Customer ID 9 (RECENT_HIGH_VALUE):**
    -   **Action:** Standard personalized offer.
    -   **Reason Code:** `RECENT_HIGH_VALUE` (Recency: 8 days, Avg Purchase: $90.0).
    -   **Confidence Note:** High confidence. This customer is very recent and has a solid average purchase value.
    -   **What would make it wrong:** If the customer has recently switched to a competitor or has expressed dissatisfaction in other channels not captured by this data.

-   **Customer ID 19 (RECENT_HIGH_VALUE):**
    -   **Action:** Standard personalized offer.
    -   **Reason Code:** `RECENT_HIGH_VALUE` (Recency: 18 days, Avg Purchase: $100.0).
    -   **Confidence Note:** High confidence. Good recency and a high average purchase value.
    -   **What would make it wrong:** If this customer is already overwhelmed with communications from the brand and adding another might lead to opt-out.

### General Considerations:

For `RECENT_LOW_VALUE` customers (e.g., Customer IDs 2, 4, 11, 16), the action might be a re-engagement offer with a lower discount, aiming to increase their purchase value. While they are active, their lower spending indicates they might need a different incentive.

The confidence in these picks is based solely on the two features considered (recency and average purchase value). External factors, not included in this simple rule, could significantly alter the actual effectiveness of an action.

In [ ]:
# For demonstration, let's create a synthetic dataset for customer activity.
# In a real scenario, this data would come from a database or a file.

# Set a fixed 'current date' for consistent recency calculation
CURRENT_DATE = pd.to_datetime('2023-10-26')

customer_data = {
    'customer_id': range(1, 21), # 20 customers
    'last_activity_date': [
        '2023-10-20', '2023-10-01', '2023-09-15', '2023-10-25', '2023-08-01',
        '2023-10-10', '2023-09-28', '2023-07-01', '2023-10-18', '2023-09-01',
        '2023-10-22', '2023-09-05', '2023-08-10', '2023-10-05', '2023-07-20',
        '2023-10-24', '2023-09-10', '2023-08-15', '2023-10-08', '2023-07-25'
    ],
    'average_purchase_value': [
        60.0, 30.0, 75.0, 10.0, 120.0, # RECENT_HIGH, RECENT_LOW, OLD_HIGH, RECENT_LOW, OLD_HIGH
        80.0, 45.0, 200.0, 90.0, 25.0,  # RECENT_HIGH, OLD_LOW, OLD_HIGH, RECENT_HIGH, OLD_LOW
        15.0, 110.0, 55.0, 70.0, 40.0,  # RECENT_LOW, OLD_HIGH, OLD_HIGH, RECENT_HIGH, OLD_LOW
        5.0, 150.0, 35.0, 100.0, 10.0   # RECENT_LOW, OLD_HIGH, OLD_LOW, RECENT_HIGH, OLD_LOW
    ]
}

df = pd.DataFrame(customer_data)
df['last_activity_date'] = pd.to_datetime(df['last_activity_date'])

# Calculate recency
df['recency_days'] = (CURRENT_DATE - df['last_activity_date']).dt.days

# Define a function to calculate the action score and reason code
def calculate_action_score_and_reason(row):
    is_recent = row['recency_days'] <= RECENT_ACTIVITY_THRESHOLD_DAYS
    is_high_value = row['average_purchase_value'] >= AVERAGE_PURCHASE_VALUE_THRESHOLD

    score = 0
    reason_code = ''

    if is_recent and is_high_value:
        score = 100 # Highest score
        reason_code = 'RECENT_HIGH_VALUE'
    elif is_recent and not is_high_value:
        score = 70 # Medium-high score
        reason_code = 'RECENT_LOW_VALUE'
    elif not is_recent and is_high_value:
        score = 40 # Medium-low score
        reason_code = 'OLD_HIGH_VALUE'
    elif not is_recent and not is_high_value:
        score = 10 # Lowest score
        reason_code = 'OLD_LOW_VALUE'

    return pd.Series([score, reason_code])

df[['action_score', 'reason_code']] = df.apply(calculate_action_score_and_reason, axis=1)

# Rank customers by action score (descending)
df['rank'] = df['action_score'].rank(method='dense', ascending=False).astype(int)

# Sort by rank and then by average_purchase_value for ties (descending)
df_ranked = df.sort_values(by=['action_score', 'average_purchase_value'], ascending=[False, False]).reset_index(drop=True)

# Create the output directory if it doesn't exist
output_dir = 'work/outputs'
os.makedirs(output_dir, exist_ok=True)

# Define the output file path
output_csv_path = os.path.join(output_dir, 'baseline_action_score.csv')

# Save the ranked queue to CSV
df_ranked.to_csv(output_csv_path, index=False)

print(f"Ranked queue saved to: {output_csv_path}")
print("\nFirst 5 rows of the ranked queue:")
print(df_ranked.head())

Ranked queue saved to: work/outputs/baseline_action_score.csv

First 5 rows of the ranked queue:
   customer_id last_activity_date  average_purchase_value  recency_days  \
0           19         2023-10-08                   100.0            18   
1            9         2023-10-18                    90.0             8   
2            6         2023-10-10                    80.0            16   
3           14         2023-10-05                    70.0            21   
4            1         2023-10-20                    60.0             6   

   action_score        reason_code  rank  
0           100  RECENT_HIGH_VALUE     1  
1           100  RECENT_HIGH_VALUE     1  
2           100  RECENT_HIGH_VALUE     1  
3           100  RECENT_HIGH_VALUE     1  
4           100  RECENT_HIGH_VALUE     1  


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak Picks Analysis:

**Which picks look wrong and why?**

The 'weak picks' in this model are generally those with lower action scores, particularly those categorized as `OLD_LOW_VALUE` and `OLD_HIGH_VALUE`.

-   **`OLD_LOW_VALUE` customers (e.g., Customer IDs 20, 2, 7, 10, 15, 18):** These customers have not engaged recently and have a low average purchase value. While they still receive a score, the likelihood of a positive response to an action is low. Investing heavily in engaging these customers might not yield a good return on investment. Their inclusion in a general action queue might be inefficient.
    -   **Why they might be 'wrong' picks:** They may be churned customers, have left the market, or simply not be interested in the product/service anymore. Without more data on their current status, targeting them might be a wasted effort.

-   **`OLD_HIGH_VALUE` customers (e.g., Customer IDs 5, 8, 12, 17):** These customers have a high average purchase value but their last activity was not recent. While they historically spent more, their lack of recent engagement makes them 'weaker' than recent customers. They might represent dormant high-value customers who require a specific re-activation strategy rather than a standard offer.
    -   **Why they might be 'wrong' picks for *general* campaigns:** A generic offer might not be enough to re-engage them. A more tailored approach, perhaps based on their past purchase history, might be more effective. Treating them the same as `RECENT_HIGH_VALUE` customers ignores their disengagement.

### Leakage Check:

**Confirm no product flags or future windows leaked in.**

In this baseline model, we have been careful to avoid common leakage issues:

1.  **Product Flags:** The synthetic dataset does not include any explicit 'product flags' that might implicitly reveal a customer's responsiveness. The features used (`last_activity_date`, `average_purchase_value`) are derived solely from historical activity.

2.  **Future Windows:** The `CURRENT_DATE` was explicitly set to `2023-10-26`, and all 'last_activity_date' values are strictly prior to or on this date. The `recency_days` calculation is based on the difference between the `CURRENT_DATE` and the `last_activity_date`, ensuring that no future information is used to determine recency. All calculations are performed using historical data relative to the defined `CURRENT_DATE`.

3.  **Action Outcome Information:** The dataset does not contain any information about whether a customer *responded* to a previous action. The scores and reason codes are generated purely based on the defined rule and available historical customer attributes.

Therefore, based on the construction of the synthetic dataset and the rule, there appears to be no leakage of product flags or future window information.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.